Bill of Quantities - PV Minas do Sol
=
This notebook counts the tracker structures and measures the trenches, ducts and cables drawn in the Minas do Sol DC drawing, and writes the quantities to `C:\Users\Usuario\Desktop\Minas do Sol - BOQ.xlsx`. Lengths are summed **per layer**: the layer name tells the item type and, through its suffix, the number of circuits (`NNxC`, `NNC`) or conduits (`NS`).

**Workflow overview:**
1. Convert the DWG to DXF and load all model space entities into `pv_projects.duckdb` (project `"PV Minas do Sol"`).
2. Select the entities of the project as a DuckDB relation (`data`).
3. **Structures:** take every string/tracker block, group the blocks into tracker columns (same X) and split each column where there is a gap in Y. The number of groups is the number of structures.
4. **Cables:** select AC trenches, DC cables, DC ducts and DC trenches by layer name, then sum `length` per layer and count the ducts.
5. Write every summary table to its own sheet and close the connection.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `Minas do Sol - DC - leapfrog(1).dwg` in `C:\Users\Usuario\Desktop\projetos\Minas`. The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** blocks and lines inside other blocks, xrefs or paper space aren't read.
- **Tracker blocks:**
  - One block reference (`INSERT`) per string, with a name starting with `String_Tracker_` (e.g. `String_Tracker_1X174p_1`). `LIKE` is case-sensitive, `_` matches any single character, and no other block may start with this text.
  - The trackers run north-south: all the string blocks of one tracker have the same insertion X (rounded to the metre), and consecutive strings of a tracker are at most 37.5 m apart in Y.
  - Two trackers in the same column must be more than 37.5 m apart in Y, otherwise they are counted as one structure.
- **Entity types:** trenches, ducts and cables must be `LINE` or `LWPOLYLINE` (only these get a `length`). Drawing units are metres.
- **Layer names** (SQL `LIKE`, case-sensitive):
  - AC trenches: `EMF_bt_vala_btca_<n>xC` (none in the current drawing: the sheet comes out empty).
  - DC cables: exactly `EMF_cabo_solar_negativo` and `EMF_cabo_solar_positivo`.
  - DC ducts: `EMF_bt_eletroduto_<n>S`.
  - DC trenches: `EMF_vala_CC_<n>C`.
- **Ducts are counted per entity:** each duct piece must be one single LINE or LWPOLYLINE.

Extracting the AutoCAD data and loading it into the DuckDB database
=
- Copies `Minas do Sol - DC - leapfrog(1).dwg` from `dwg_folder` into `temp_folder`, so the ODA File Converter only converts that one file and not every DWG in the project folder.
- `convert_dwg_to_dxf` (from `DWG_to_DuckDB_ETL.py`) writes the DXF (ACAD2018) into `output_folder`, which is the same folder as the DWG.
- `load_to_duckdb` reads every model space entity with `ezdxf` and loads it into the `cad_entities` table under the project name `"PV Minas do Sol"`. If the project already exists, its old rows are deleted first, so the table always holds the latest version of the drawing.
  - `LINE` / `LWPOLYLINE`: start/end point and length (for polylines, along all segments including bulge arcs).
  - `INSERT` (block reference): block name, insertion point (`X`, `Y`), rotation, color and the attribute values.
- At the end, the temporary DXF and the copied DWG are deleted. The DuckDB connection (`conn`) stays open for the rest of the notebook and is closed in the last cell.
- `rewrite=True` is the default of `load_to_duckdb`, it's only written out here.

In [1]:
import pandas as pd

from DWG_to_DuckDB_ETL import *
import os
import shutil

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\Minas"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\Minas\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "Minas do Sol - DC - leapfrog(1).dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "Minas do Sol - DC - leapfrog(1).dxf")

temp_dwg_file = os.path.join(temp_folder, "Minas do Sol - DC - leapfrog(1).dwg")

conn = load_to_duckdb(dxf_file, "PV Minas do Sol", rewrite=True)
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Minas do Sol already exists
Removed existing data for project: PV Minas do Sol
Successfully loaded project
Loaded 48718 entities for project: PV Minas do Sol
Don't forget to close connection.
Data Loaded
DuckDB Loaded


CABLE AND STRUCTURE DATA
=

- `data` is a DuckDB relation with `name`, `layer`, insertion point (`X`, `Y`), start/end coordinates and `length` of project `"PV Minas do Sol"`. The next cells query it by its Python variable name (DuckDB can query Python relations/DataFrames by name).
- `DESCRIBE cad_entities` prints the table schema.
- The layer count runs on `data`, so it only shows the layers of this drawing. Use it to check the layer names the next cells filter on.

In [2]:
data = conn.sql("SELECT name, layer, X, Y, start_x, start_y, end_x, end_y, length FROM cad_entities WHERE project_name='PV Minas do Sol'")
print(data)
print(conn.execute("DESCRIBE cad_entities").fetchdf())

result = conn.execute("""
    SELECT
        layer,
        COUNT(*) as count
    FROM data
    GROUP BY layer
    ORDER BY count DESC
""").fetchdf()
print(result)

┌───────────────────────────┬─────────┬───────────┬───────────┬─────────┬─────────┬───────┬───────┬────────┐
│           name            │  layer  │     X     │     Y     │ start_x │ start_y │ end_x │ end_y │ length │
│          varchar          │ varchar │   float   │   float   │  float  │  float  │ float │ float │ float  │
├───────────────────────────┼─────────┼───────────┼───────────┼─────────┼─────────┼───────┼───────┼────────┤
│ String_Tracker_1X174p_1   │ MDS_1.1 │  511819.7 │ 8087402.5 │    NULL │    NULL │  NULL │  NULL │   NULL │
│ String_Tracker_1X174p_2   │ MDS_1.1 │  511819.7 │ 8087368.5 │    NULL │    NULL │  NULL │  NULL │   NULL │
│ String_Tracker_1X174p_3   │ MDS_1.1 │  511819.7 │ 8087334.5 │    NULL │    NULL │  NULL │  NULL │   NULL │
│ String_Tracker_1X174p_1   │ MDS_1.1 │  511819.7 │ 8087230.5 │    NULL │    NULL │  NULL │  NULL │   NULL │
│ String_Tracker_1X174p_2   │ MDS_1.1 │  511819.7 │ 8087264.5 │    NULL │    NULL │  NULL │  NULL │   NULL │
│ String_Tracker_1X

STRUCTURE DATA

The next two cells count the tracker structures from the string blocks.

Selecting the tracker blocks
=
Every block reference whose name starts with `String_Tracker_` (one per string). Only the layer and the insertion point are kept.

In [3]:
structures = conn.sql("SELECT layer, X, Y FROM data WHERE name LIKE 'String_Tracker_%'")
print(structures)

┌─────────┬──────────┬───────────┐
│  layer  │    X     │     Y     │
│ varchar │  float   │   float   │
├─────────┼──────────┼───────────┤
│ MDS_1.1 │ 511819.7 │ 8087402.5 │
│ MDS_1.1 │ 511819.7 │ 8087368.5 │
│ MDS_1.1 │ 511819.7 │ 8087334.5 │
│ MDS_1.1 │ 511819.7 │ 8087230.5 │
│ MDS_1.1 │ 511819.7 │ 8087264.5 │
│ MDS_1.1 │ 511819.7 │ 8087298.5 │
│ MDS_1.1 │ 511812.9 │ 8087402.5 │
│ MDS_1.1 │ 511812.9 │ 8087368.5 │
│ MDS_1.1 │ 511812.9 │ 8087334.5 │
│ MDS_1.1 │ 511812.9 │ 8087230.5 │
│    ·    │     ·    │     ·     │
│    ·    │     ·    │     ·     │
│    ·    │     ·    │     ·     │
│ MDS_4.1 │ 513646.2 │ 8088546.5 │
│ MDS_4.1 │ 513646.2 │ 8088580.5 │
│ MDS_4.1 │ 513646.2 │ 8088614.5 │
│ MDS_4.1 │ 513768.6 │ 8088718.5 │
│ MDS_4.1 │ 513768.6 │ 8088684.5 │
│ MDS_4.1 │ 513768.6 │ 8088650.5 │
│ MDS_4.1 │ 513768.6 │ 8088546.5 │
│ MDS_4.1 │ 513768.6 │ 8088580.5 │
│ MDS_4.1 │ 513768.6 │ 8088614.5 │
│ MDS_4.1 │ 513721.0 │ 8088718.5 │
└─────────┴──────────┴───────────┘
  ? rows            

Counting the structures
=
- `Line` = insertion X rounded to the metre: the tracker column the string belongs to.
- Sort order: `Line` ascending, then `Y` descending (column by column, north to south).
- A new structure starts when:
  1. the column changes (`Line` differs from the previous row), or
  2. the Y gap to the previous string is larger than **37.5 m** (the strings of one tracker are ~34 m apart, so a larger gap means the next tracker).
- `Group` = cumulative sum of the break flags, so each tracker gets its own number and the maximum is the number of structures.
- The count is also stored in the one-row DataFrame `number_of_structures`, exported to the `Number of Structures` sheet.
- The comment "no C type on the edges" is a leftover from another notebook: there are no structure types here.
- Note: X is rounded to the whole metre, so a column whose X sits close to `.5` can be split into two columns.

In [4]:
structure_df = structures.fetchdf()
structure_df['Line'] = round(structure_df['X'],0)

structure_df.sort_values(by=['Line', 'Y'], ascending=[True, False], inplace=True, ignore_index=True)

#Different Lines
condition1 = structure_df["Line"] != structure_df["Line"].shift()

#Break in the Structure with no C type on the edges
condition2 = (abs(structure_df["Y"] - structure_df["Y"].shift()) > 37.5)


break_condition = condition1 | condition2

structure_df['Group'] = break_condition.cumsum()
print(f'The number of structures is: {max(structure_df['Group'])}')

number_of_structures = pd.DataFrame({'Number of Structures': max(structure_df['Group'])}, index=[0])
print(number_of_structures)

The number of structures is: 1765
   Number of Structures
0                  1765


CABLE DATA

The next cells measure the trenches, ducts and cables by layer.

Splitting the cable entities into groups
=
One relation per group, selected from `data` by layer name:
- `ac_cables`: `LIKE 'EMF_bt_vala_btca_%xC'` (AC trenches).
- `dc_cables`: exact match on the positive and negative solar cable layers.
- `dc_ducts`: `LIKE 'EMF_bt_eletroduto_%S'`.
- `dc_trenches`: `LIKE 'EMF_vala_CC_%C'`.

`LIKE` is case-sensitive. Layers that match no pattern are ignored. There are no watercourse/access ductbank groups in this project.

In [5]:
ac_cables = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_vala_btca_%xC'
""")

dc_cables = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer = 'EMF_cabo_solar_negativo' OR layer = 'EMF_cabo_solar_positivo'
""")

dc_ducts = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_eletroduto_%S'
""")

dc_trenches = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_vala_CC_%C'
""")

QTY. OF AC CABLES (m)
=

Prints the AC trench entities (0 rows in the current drawing).

In [6]:
print(ac_cables)

┌─────────┬─────────┬─────────┬───────┬───────┬────────┐
│  layer  │ start_x │ start_y │ end_x │ end_y │ length │
│ varchar │  float  │  float  │ float │ float │ float  │
└─────────┴─────────┴─────────┴───────┴───────┴────────┘
                         0 rows                       



AC trench length per layer
=
`SUM(length)` per layer, sorted by layer name. The relation is referenced by its variable name (here written as a string). Goes to the `AC Trenches (m)` sheet. Empty for the current drawing, since it has no AC trench layers.

In [7]:
ac_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'ac_cables' GROUP BY layer ORDER BY layer")
print(ac_cables_group)

┌─────────┬──────────────┐
│  layer  │ total_length │
│ varchar │    double    │
└─────────┴──────────────┘
          0 rows        



QTY. OF DC CABLES (m)
=

Prints every positive and negative solar cable run.

In [8]:
print(dc_cables)

┌─────────────────────────┬──────────┬───────────┬───────────┬───────────┬───────────┐
│          layer          │ start_x  │  start_y  │   end_x   │   end_y   │  length   │
│         varchar         │  float   │   float   │   float   │   float   │   float   │
├─────────────────────────┼──────────┼───────────┼───────────┼───────────┼───────────┤
│ EMF_cabo_solar_positivo │ 511778.9 │ 8087248.5 │  511791.8 │ 8087246.0 │  13.90625 │
│ EMF_cabo_solar_positivo │ 511778.9 │ 8087246.5 │  511791.8 │ 8087246.0 │  16.19727 │
│ EMF_cabo_solar_positivo │ 511785.7 │ 8087386.5 │  511791.8 │ 8087246.0 │   145.125 │
│ EMF_cabo_solar_positivo │ 511785.7 │ 8087352.5 │  511791.8 │ 8087246.0 │   111.125 │
│ EMF_cabo_solar_positivo │ 511785.7 │ 8087317.5 │  511791.8 │ 8087246.0 │    76.125 │
│ EMF_cabo_solar_positivo │ 511785.7 │ 8087282.5 │  511791.8 │ 8087246.0 │    41.125 │
│ EMF_cabo_solar_positivo │ 511785.7 │ 8087248.5 │  511791.8 │ 8087246.0 │     7.125 │
│ EMF_cabo_solar_positivo │ 511792.5 │ 8087

DC cable length per layer and total
=
- Per-layer `SUM(length)` for the positive and the negative layers.
- The grand total (both polarities) is a second query, rounded to 3 decimals. Only the per-layer table is exported (`DC Cables (m)`).
- Overlapping identical lines are all counted: nothing removes duplicates, so every cable drawn on top of another one is summed again.

In [9]:
dc_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_cables' GROUP BY layer ORDER BY layer")
print(dc_cables_group)

print(f'The total length of DC cables is {round(conn.sql("SELECT SUM(length) FROM 'dc_cables'").fetchone()[0],3)} meters')

┌─────────────────────────┬───────────────────┐
│          layer          │   total_length    │
│         varchar         │      double       │
├─────────────────────────┼───────────────────┤
│ EMF_cabo_solar_negativo │ 1157384.073344946 │
│ EMF_cabo_solar_positivo │ 1157384.073344946 │
└─────────────────────────┴───────────────────┘

The total length of DC cables is 2314768.147 meters


QTY. OF DC DUCTS (m and un.)
=

Prints the DC duct entities. (The longest-cable check right after it belongs to the DC cables.)

In [10]:
print(dc_ducts)

┌──────────────────────┬───────────┬───────────┬────────────┬───────────┬───────────┐
│        layer         │  start_x  │  start_y  │   end_x    │   end_y   │  length   │
│       varchar        │   float   │   float   │   float    │   float   │   float   │
├──────────────────────┼───────────┼───────────┼────────────┼───────────┼───────────┤
│ EMF_bt_eletroduto_1S │  511690.5 │ 8087214.5 │   511703.4 │ 8087212.0 │  13.90625 │
│ EMF_bt_eletroduto_1S │  511710.9 │ 8087214.5 │  511704.78 │ 8087212.0 │     7.125 │
│ EMF_bt_eletroduto_1S │  511758.5 │ 8087214.5 │   511771.4 │ 8087212.0 │  13.90625 │
│ EMF_bt_eletroduto_1S │  511778.9 │ 8087214.5 │  511772.78 │ 8087212.0 │     7.125 │
│ EMF_bt_eletroduto_1S │  511853.7 │ 8087214.5 │   511859.8 │ 8087212.0 │     7.125 │
│ EMF_bt_eletroduto_1S │  511874.1 │ 8087214.5 │   511861.2 │ 8087212.0 │  13.90625 │
│ EMF_bt_eletroduto_1S │  511921.7 │ 8087214.5 │   511927.8 │ 8087212.0 │     7.125 │
│ EMF_bt_eletroduto_1S │  511670.1 │ 8087193.5 │   511

Longest DC cables
=
Sanity check only: prints the 12 longest DC cable runs (`ORDER BY length DESC LIMIT 12`), to spot a cable that was drawn to the wrong inverter/combiner. Nothing is exported.

The positive and negative cables are listed in pairs, as they follow the same route.

In [11]:
longest_DC_cable = conn.sql('SELECT * FROM dc_cables ORDER BY length DESC LIMIT 12')
print(longest_DC_cable) #Longest DC cable

┌─────────────────────────┬────────────┬───────────┬───────────┬───────────┬───────────┐
│          layer          │  start_x   │  start_y  │   end_x   │   end_y   │  length   │
│         varchar         │   float    │   float   │   float   │   float   │   float   │
├─────────────────────────┼────────────┼───────────┼───────────┼───────────┼───────────┤
│ EMF_cabo_solar_positivo │   512933.7 │ 8088376.5 │  512940.5 │ 8088129.0 │  252.6738 │
│ EMF_cabo_solar_negativo │   512933.7 │ 8088376.5 │  512940.5 │ 8088129.0 │  252.6738 │
│ EMF_cabo_solar_positivo │  512109.47 │ 8087248.5 │ 512089.75 │ 8087421.5 │ 223.76848 │
│ EMF_cabo_solar_negativo │  512109.47 │ 8087248.5 │ 512089.75 │ 8087421.5 │ 223.76848 │
│ EMF_cabo_solar_positivo │   512933.7 │ 8088342.5 │  512940.5 │ 8088129.0 │  218.6738 │
│ EMF_cabo_solar_negativo │   512933.7 │ 8088342.5 │  512940.5 │ 8088129.0 │  218.6738 │
│ EMF_cabo_solar_positivo │ 515440.875 │ 8087717.0 │ 515453.78 │ 8087863.5 │ 209.67902 │
│ EMF_cabo_solar_nega

DC duct length per layer
=
Per-layer sum for `1S` / `2S` / `3S` (the suffix is the number of conduits/strings in the duct). Goes to the `DC Ducts (m)` sheet.

In [12]:
dc_ducts_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_ducts' GROUP BY layer ORDER BY layer")
print(dc_ducts_group)

┌──────────────────────┬────────────────────┐
│        layer         │    total_length    │
│       varchar        │       double       │
├──────────────────────┼────────────────────┤
│ EMF_bt_eletroduto_1S │  3844.093280315399 │
│ EMF_bt_eletroduto_2S │  7557.504392147064 │
│ EMF_bt_eletroduto_3S │ 23894.632987499237 │
└──────────────────────┴────────────────────┘



DC ducts: number of units
=
Number of duct entities per layer and in total. Each LINE/LWPOLYLINE on a duct layer counts as one duct. Goes to the `DC Ducts (un.)` sheet.

In [13]:
dc_ducts_count = conn.sql("SELECT layer, COUNT(layer) as count FROM 'dc_ducts' GROUP BY layer ORDER BY count DESC")
print(dc_ducts_count)

print(f'The total quantity of DC ducts is {conn.sql("SELECT COUNT(layer) FROM 'dc_ducts'").fetchone()[0]}')

┌──────────────────────┬───────┐
│        layer         │ count │
│       varchar        │ int64 │
├──────────────────────┼───────┤
│ EMF_bt_eletroduto_3S │  3719 │
│ EMF_bt_eletroduto_2S │   635 │
│ EMF_bt_eletroduto_1S │   341 │
└──────────────────────┴───────┘

The total quantity of DC ducts is 4695


QTY. OF DC TRENCHES (m)
=

Prints the DC trench entities.

In [14]:
print(dc_trenches)

┌─────────────────┬────────────┬───────────┬────────────┬───────────┬───────────┐
│      layer      │  start_x   │  start_y  │   end_x    │   end_y   │  length   │
│     varchar     │   float    │   float   │   float    │   float   │   float   │
├─────────────────┼────────────┼───────────┼────────────┼───────────┼───────────┤
│ EMF_vala_CC_5C  │   512936.3 │ 8088236.0 │   512940.5 │ 8088236.0 │    4.1875 │
│ EMF_vala_CC_1C  │  512320.25 │ 8087250.5 │ 512317.625 │ 8087248.0 │     3.625 │
│ EMF_vala_CC_1C  │  512197.84 │ 8087196.0 │  512200.47 │ 8087198.5 │     3.625 │
│ EMF_vala_CC_1C  │  512225.06 │ 8087196.0 │  512222.44 │ 8087198.5 │     3.625 │
│ EMF_vala_CC_1C  │  512265.84 │ 8087196.0 │  512268.47 │ 8087198.5 │     3.625 │
│ EMF_vala_CC_1C  │  512293.06 │ 8087196.0 │  512290.44 │ 8087198.5 │     3.625 │
│ EMF_vala_CC_1C  │  512333.84 │ 8087196.0 │  512336.47 │ 8087198.5 │     3.625 │
│ EMF_vala_CC_1C  │  512361.06 │ 8087196.0 │  512358.44 │ 8087198.5 │     3.625 │
│ EMF_vala_CC_1C

DC trench length per layer and total
=
Per-layer sum of the `EMF_vala_CC_<n>C` layers (`<n>C` = number of DC circuits in the trench), plus the grand total. Goes to the sheet named `DC Trenches (un.)`, although the values are lengths in metres.

In [15]:
dc_trenches_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_trenches' GROUP BY layer ORDER BY layer")
print(dc_trenches_group)

print(f'\nThe total length of DC trenches is {round(conn.sql("SELECT SUM(length) FROM 'dc_trenches'").fetchone()[0],3)} meters')

┌─────────────────┬────────────────────┐
│      layer      │    total_length    │
│     varchar     │       double       │
├─────────────────┼────────────────────┤
│ EMF_vala_CC_10C │  846.3577001690865 │
│ EMF_vala_CC_11C │            3.46875 │
│ EMF_vala_CC_12C │            2.96875 │
│ EMF_vala_CC_13C │            3.46875 │
│ EMF_vala_CC_14C │                3.5 │
│ EMF_vala_CC_1C  │  258.3968994617462 │
│ EMF_vala_CC_2C  │  3441.166440963745 │
│ EMF_vala_CC_3C  │  310.8299083709717 │
│ EMF_vala_CC_4C  │ 2844.2146011590958 │
│ EMF_vala_CC_5C  │ 1769.1333198547363 │
│ EMF_vala_CC_6C  │  6942.574889063835 │
│ EMF_vala_CC_7C  │  113.8006477355957 │
│ EMF_vala_CC_8C  │ 1204.6789104938507 │
│ EMF_vala_CC_9C  │            87.1875 │
└─────────────────┴────────────────────┘
  14 rows                    2 columns


The total length of DC trenches is 17831.747 meters


OVERVIEW
=

Writes every summary to its own sheet of `C:\Users\Usuario\Desktop\Minas do Sol - BOQ.xlsx` (hardcoded path, overwritten on every run), then closes the DuckDB connection.
- `AC Trenches (m)` is written even when it's empty.
- The `DC Trenches (un.)` sheet holds lengths in metres, not units.

In [16]:
with pd.ExcelWriter(r"C:\Users\Usuario\Desktop\Minas do Sol - BOQ.xlsx") as xlsx:
    ac_cables_group.df().to_excel(xlsx, sheet_name='AC Trenches (m)', index=False)
    dc_cables_group.df().to_excel(xlsx, sheet_name='DC Cables (m)', index=False)
    dc_ducts_group.df().to_excel(xlsx, sheet_name='DC Ducts (m)', index=False)
    dc_ducts_count.df().to_excel(xlsx, sheet_name='DC Ducts (un.)', index=False)
    dc_trenches_group.df().to_excel(xlsx, sheet_name='DC Trenches (un.)', index=False)
    number_of_structures.to_excel(xlsx, sheet_name='Number of Structures', index=False)


conn.close()
